# Phase 1 - Validation Suite: State U3 Unemployment Rate (LAUS)

**Scope:** validate the state headline (U3) unemployment-rate dataset (`{ABBR}UR`, monthly seasonally adjusted, percent, BLS Local Area Unemployment Statistics via FRED/ALFRED) before any modeling. State labor force (`{ABBR}LF`) is pulled for the labor-force-weighted reconciliation; national UNRATE for the benchmark.

**Downstream uses and what they stress:**

| Use case | Failure modes probed here |
|---|---|
| Forecasting | persistence/stationarity of a bounded rate, residual seasonality, outliers (V5, V7) |
| Lead-lag | timestamp & alignment conventions, sign orientation vs payrolls (V3, V8) |
| Backtesting | point-in-time integrity, benchmark revisions, publication lag, look-ahead guards (V3, V4, V9) |

**Where U3 sits among the three state datasets.** This suite is the sibling of the state continued-claims and nonfarm-payroll suites; U3 is a *third distinct profile* and the checks are tuned to it:

| | Continued claims | Nonfarm payrolls | **U3 unemployment** |
|---|---|---|---|
| Data type | level (weekly count) | level (monthly count, thousands) | **rate (percent, bounded)** |
| Frequency | weekly | monthly | **monthly** (ref. week incl. the 12th) |
| Seasonal adjustment | NSA | SA | **SA** (seasonality should be absent) |
| Revised after first print? | ~0% | ~99% | **moderate** (~74%, but quantized to 0.1pp; annual benchmark) |
| Level behaviour | cyclical | trending (I(1)) | **cyclical, bounded, no trend** |
| Reconciliation with national | sum (PR/VI gap) | sum (~exact) | **labor-force-weighted mean** (rates cannot be summed) |
| Publication lag | ~10 days | ~50 days | **~50 days** (same joint State Employment & Unemployment report as NFP) |

The two headline consequences for U3: (1) it is a **bounded rate**, so features are *changes in percentage points*, not growth rates, and long-run mean-reversion (not a trend) governs stationarity; (2) reconciliation requires **weighting by labor force** - the naive average of state rates is biased. Each check states its question and pass criterion up front; the last cells render the report and go/no-go gates.

In [ ]:
import os, time, json, hashlib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FRED_API_KEY = os.getenv("FRED_API_KEY", "12d77a40907e43a92e9a295801db18d2")
FRED_URL     = "https://api.stlouisfed.org/fred/series/observations"

DATA_FILES = {
    "vintage": "ur_vintage_first_release.csv",
    "current": "ur_current_history.csv",
    "pit":     "ur_pit_features.csv",
}

checks = []

def add_check(module, check_id, name, result, metric="", threshold="", note=""):
    assert result in ("PASS", "WARN", "FAIL")
    checks.append(dict(module=module, id=check_id, check=name, result=result,
                       metric=str(metric), threshold=str(threshold), note=note))
    print(f"[{result:4s}] {check_id:6s} {name}  |  {metric}  (criterion: {threshold})")
    if note:
        print(f"       note: {note}")


def fred_get(params, retries=5):
    params = {**params, "api_key": FRED_API_KEY, "file_type": "json"}
    for attempt in range(retries):
        data = requests.get(FRED_URL, params=params, timeout=180).json()
        if "error_message" in data:
            if "Too Many Requests" in data["error_message"]:
                time.sleep(2 ** (attempt + 1)); continue
            raise ValueError(data["error_message"])
        return data
    raise RuntimeError(f"Rate limit persists: {params.get('series_id')}")


def fred_obs(params, retries=5):
    return [o for o in fred_get(params, retries).get("observations", []) if o["value"] != "."]

print("Setup ok.")

## Maintained hypotheses - LAUS publication mechanics

**H1 - Publication schedule.** The state U3 unemployment rate is published in the BLS *State Employment and Unemployment* report - the **same monthly release as state nonfarm payrolls**, roughly **three weeks after** the national Employment Situation, about **7 weeks (~50 days) after the reference month**, on a Friday.
→ tested by **V3.2** (national release weekday), **V3.3** (state publication lag ≈ 50 days), **V3.4** (follows the national jobs report).

**H2 - Reference period.** The rate reflects labor-force status in the **week containing the 12th** of the reference month; FRED stamps the observation on the **1st of that month**.
→ tested by **V2.1** (all observation dates are month-starts).

**H3 - Revision structure.** LAUS estimates are re-estimated and **benchmarked annually** (re-anchored to CES/QCEW inputs and re-seasonally-adjusted each winter). Revisions are moderate and **quantized to 0.1 percentage points** because the rate is published to one decimal - so many true revisions round to zero.
→ quantified by **V4** (first-vs-current revision, benchmark clustering) and **V9** (feature contamination).

**Vocabulary.** *First release* = the initial estimate for a month (its `first_release_date`). *Current* = the latest value after all revisions and benchmarks. *Two clocks* = observation time (the month measured, V2) vs information time (when it became known, V3). Because the rate is bounded, feature transforms are **changes in percentage points** (`ur_t0 - ur_t12`), not percent changes.

## V0 - Build the dataset from the FRED/ALFRED API (self-contained)

**V0a** pulls, per state, the UR ALFRED first-release table (`output_type=4`), the UR current history and the LF (labor force) current history (~155 calls, ~3-4 min). **V0b** assembles the monthly point-in-time panel, persists three CSVs and pulls national UNRATE. Set `USE_CACHE = True` to reuse a previous run's CSVs.

In [ ]:
# ── V0a. Fetch state U3 unemployment + labor force from the FRED/ALFRED API ────
USE_CACHE = False   # True -> reuse CSVs from a previous run (skips the ~3-4 min fetch)

STATES = [
    ("Alabama","AL","01"),("Alaska","AK","02"),("Arizona","AZ","04"),("Arkansas","AR","05"),
    ("California","CA","06"),("Colorado","CO","08"),("Connecticut","CT","09"),("Delaware","DE","10"),
    ("District of Columbia","DC","11"),("Florida","FL","12"),("Georgia","GA","13"),("Hawaii","HI","15"),
    ("Idaho","ID","16"),("Illinois","IL","17"),("Indiana","IN","18"),("Iowa","IA","19"),
    ("Kansas","KS","20"),("Kentucky","KY","21"),("Louisiana","LA","22"),("Maine","ME","23"),
    ("Maryland","MD","24"),("Massachusetts","MA","25"),("Michigan","MI","26"),("Minnesota","MN","27"),
    ("Mississippi","MS","28"),("Missouri","MO","29"),("Montana","MT","30"),("Nebraska","NE","31"),
    ("Nevada","NV","32"),("New Hampshire","NH","33"),("New Jersey","NJ","34"),("New Mexico","NM","35"),
    ("New York","NY","36"),("North Carolina","NC","37"),("North Dakota","ND","38"),("Ohio","OH","39"),
    ("Oklahoma","OK","40"),("Oregon","OR","41"),("Pennsylvania","PA","42"),("Rhode Island","RI","44"),
    ("South Carolina","SC","45"),("South Dakota","SD","46"),("Tennessee","TN","47"),("Texas","TX","48"),
    ("Utah","UT","49"),("Vermont","VT","50"),("Virginia","VA","51"),("Washington","WA","53"),
    ("West Virginia","WV","54"),("Wisconsin","WI","55"),("Wyoming","WY","56"),
]
# Series: {ABBR}UR = SA U3 rate (validated series); {ABBR}LF = SA labor force (weights for reconciliation)

def fetch_vintage(series_id):
    return fred_obs({"series_id": series_id, "realtime_start": "1776-07-04",
                     "realtime_end": "9999-12-31", "output_type": 4, "limit": 100000})
def fetch_current(series_id):
    return fred_obs({"series_id": series_id, "observation_start": "1976-01-01", "limit": 100000})


if USE_CACHE and all(os.path.exists(f) for f in DATA_FILES.values()):
    df_v = pd.read_csv(DATA_FILES["vintage"], parse_dates=["obs_date", "first_release_date"])
    df_c = pd.read_csv(DATA_FILES["current"], parse_dates=["obs_date"])
    pit  = pd.read_csv(DATA_FILES["pit"], parse_dates=["as_of_date", "ur_latest_obs"])
    print("Loaded cached CSVs (USE_CACHE=True).")
else:
    vintage_rows, current_rows, failed = [], [], []
    for i, (name, abbr, fips) in enumerate(STATES):
        try:
            for o in fetch_vintage(f"{abbr}UR"):
                vintage_rows.append({"state": name, "abbr": abbr, "fips": fips,
                    "obs_date": pd.to_datetime(o["date"]),
                    "first_release_date": pd.to_datetime(o["realtime_start"]),
                    "superseded_date": o["realtime_end"], "value_first_release": float(o["value"])})
            time.sleep(0.3)
            for o in fetch_current(f"{abbr}UR"):
                current_rows.append({"state": name, "abbr": abbr, "series": "ur",
                    "obs_date": pd.to_datetime(o["date"]), "value_current": float(o["value"])})
            time.sleep(0.3)
            for o in fetch_current(f"{abbr}LF"):
                current_rows.append({"state": name, "abbr": abbr, "series": "lf",
                    "obs_date": pd.to_datetime(o["date"]), "value_current": float(o["value"])})
            print(f"[{i+1:2d}/51] {abbr}UR / {abbr}LF ok")
        except Exception as e:
            print(f"[{i+1:2d}/51] {abbr} FAILED - {e}"); failed.append(abbr)
        time.sleep(0.3)
    if failed:
        raise RuntimeError(f"Fetch failed for {failed} - re-run this cell (rate limits are transient).")
    df_v = pd.DataFrame(vintage_rows)
    df_v["is_current"]   = df_v["superseded_date"] == "9999-12-31"
    df_v["pub_lag_days"] = (df_v["first_release_date"] - df_v["obs_date"]).dt.days
    df_v = df_v.drop(columns="superseded_date")
    df_c = pd.DataFrame(current_rows)
    pit = None

print(f"vintage rows: {len(df_v):,}   current rows: {len(df_c):,}")

In [ ]:
# ── V0b. Assemble the monthly point-in-time panel, persist, fetch national ────
N_LAGS = 13

if pit is None:
    AS_OF_GRID = pd.date_range(df_v["first_release_date"].min().to_period("M").to_timestamp(),
                               df_v["first_release_date"].max().to_period("M").to_timestamp(), freq="MS")

    def build_pit(vint, n_lags, grid):
        """Last n_lags monthly first-release readings available at each as-of month (cummax frontier)."""
        out = {}
        for state, g in vint.groupby("state", sort=False):
            g = g.sort_values("obs_date")
            rel_eff = g["first_release_date"].cummax().values
            vals = g["value_first_release"].values
            obsd = g["obs_date"].values
            n_avail = np.searchsorted(rel_eff, grid.values, side="right")
            for j, as_of in enumerate(grid):
                k = n_avail[j]
                if k < n_lags:
                    continue
                row = {f"ur_t{m}": vals[k - 1 - m] for m in range(n_lags)}
                row["ur_latest_obs"] = pd.Timestamp(obsd[k - 1])
                row["ur_lag_months"] = (as_of.to_period("M") - pd.Timestamp(obsd[k - 1]).to_period("M")).n
                out[(as_of, state)] = row
        return out

    pit = pd.DataFrame([{"as_of_date": k[0], "state": k[1], **row}
                        for k, row in build_pit(df_v, N_LAGS, AS_OF_GRID).items()])
    # A RATE: features are CHANGES in percentage points, not percent changes
    pit["ur_mom1_chg"] = pit["ur_t0"] - pit["ur_t1"]
    pit["ur_mom3_chg"] = pit["ur_t0"] - pit["ur_t3"]
    pit["ur_yoy_chg"]  = pit["ur_t0"] - pit["ur_t12"]
    df_v.to_csv(DATA_FILES["vintage"], index=False)
    df_c.to_csv(DATA_FILES["current"], index=False)
    pit.to_csv(DATA_FILES["pit"], index=False)
    print("Dataset written to CSVs.")

ur_c = df_c[df_c["series"] == "ur"].copy()   # UR current history (validated series)
lf_c = df_c[df_c["series"] == "lf"].copy()   # labor force current history (reconciliation weights)
print(f"pit rows: {len(pit):,}   UR current: {len(ur_c):,}   LF current: {len(lf_c):,}")

# National UNRATE: current + first-release table + vintage dates
nat_cur = pd.DataFrame(fred_obs({"series_id": "UNRATE", "observation_start": "1976-01-01", "limit": 100000}))
nat_cur["obs_date"] = pd.to_datetime(nat_cur["date"]); nat_cur["value"] = nat_cur["value"].astype(float)
nat_v = pd.DataFrame(fred_obs({"series_id": "UNRATE", "realtime_start": "1776-07-04",
                               "realtime_end": "9999-12-31", "output_type": 4, "limit": 100000}))
nat_v["obs_date"] = pd.to_datetime(nat_v["date"]); nat_v["first_release_date"] = pd.to_datetime(nat_v["realtime_start"])
nat_v["value"] = nat_v["value"].astype(float)
nat_v["pub_lag_days"] = (nat_v["first_release_date"] - nat_v["obs_date"]).dt.days
nat_vdates = pd.to_datetime(pd.Series(requests.get(
    "https://api.stlouisfed.org/fred/series/vintagedates",
    params={"series_id": "UNRATE", "api_key": FRED_API_KEY, "file_type": "json", "limit": 10000},
    timeout=60).json()["vintage_dates"]))
print(f"national UNRATE: {len(nat_cur)} current obs, {len(nat_v)} first releases, {len(nat_vdates)} vintage dates")

## V1 - Structural integrity

| Check | Question it answers |
|---|---|
| V1.1 | Do we have all 51 states in every table? |
| V1.2 | Is (state, series, month) a unique key? |
| V1.3 | Are key fields complete (no nulls)? |
| V1.4 | Are all rates in a plausible bounded range (0-40%)? |
| V1.5 | Are rates quantized to 0.1pp as published? |

In [ ]:
n_states = {name: df["state"].nunique() for name, df in
            [("vintage", df_v), ("UR current", ur_c), ("pit", pit)]}
add_check("V1", "V1.1", "51 states present in all tables",
          "PASS" if all(v == 51 for v in n_states.values()) else "FAIL",
          metric=n_states, threshold="== 51 everywhere")

dup_v = df_v.duplicated(["state", "obs_date"]).sum()
dup_c = df_c.duplicated(["state", "series", "obs_date"]).sum()
dup_p = pit.duplicated(["as_of_date", "state"]).sum()
add_check("V1", "V1.2", "No duplicate keys",
          "PASS" if dup_v + dup_c + dup_p == 0 else "FAIL",
          metric=f"vintage={dup_v}, current={dup_c}, pit={dup_p}", threshold="== 0")

nulls = int(df_v[["state", "obs_date", "first_release_date", "value_first_release"]].isna().sum().sum()
            + df_c[["state", "obs_date", "value_current"]].isna().sum().sum())
add_check("V1", "V1.3", "No nulls in key fields", "PASS" if nulls == 0 else "FAIL",
          metric=f"{nulls} nulls", threshold="== 0")

urv = ur_c["value_current"]
oob = int(((urv < 0) | (urv > 40)).sum() + ((df_v["value_first_release"] < 0) | (df_v["value_first_release"] > 40)).sum())
add_check("V1", "V1.4", "All rates in a plausible bounded range (0-40%)",
          "PASS" if oob == 0 else "FAIL",
          metric=f"{oob} out-of-range; observed [{urv.min():.1f}, {urv.max():.1f}]%", threshold="0 outside [0, 40]",
          note="a rate is bounded - a value near 0 or above ~30 (COVID peak) is the natural check, unlike an "
               "unbounded count")

quantized = bool(np.allclose(np.round(urv * 10), urv * 10))
add_check("V1", "V1.5", "Rates quantized to 0.1pp as published",
          "PASS" if quantized else "WARN",
          metric=f"all values 1-decimal: {quantized}", threshold="published to 0.1pp",
          note="the 0.1pp granularity means sub-0.05pp revisions round to zero (relevant in V4)")

### V1 in depth: the shape of the data

Before trusting the *content*, map the *shape*: where observations are missing, how many states report each period, whether any (state, period) key is duplicated, and - for any zero values - whether they are ever revised to non-zero. Plot titles state whether they show **first-release** or **revised (current)** data.

In [ ]:
# ── V1 shape audit: missing-value map, coverage over time, duplicate keys ─────
# Operates on the primary REVISED (current) series. Reindexing the state x period panel to the full
# regular calendar exposes any missing observation as a NaN cell.
_prim = ur_c.copy()
_states_sorted = sorted(_prim["state"].unique())
_wide_all = (_prim.pivot(index="obs_date", columns="state", values="value_current")
             .reindex(columns=_states_sorted).sort_index())
_cal = pd.date_range(_wide_all.index.min(), _wide_all.index.max(), freq="MS")
_wide_all = _wide_all.reindex(_cal)
_missing = _wide_all.isna()

# (1) heatmap of MISSING observations: states on the x-axis, time on the y-axis
fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(_missing.values, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
ax.set_xticks(range(len(_states_sorted))); ax.set_xticklabels(_states_sorted, rotation=90, fontsize=6)
_yt = np.linspace(0, len(_cal) - 1, 12).astype(int)
ax.set_yticks(_yt); ax.set_yticklabels([_cal[i].strftime("%Y-%m") for i in _yt], fontsize=7)
ax.set_xlabel("state"); ax.set_ylabel("period (time)")
ax.set_title("V1 - missing observations by state, REVISED (current) data  |  black = missing period, white = present")
plt.tight_layout(); plt.show()

# (2) counts of values reported per period over time (fields are never null per V1.3; this counts
#     how many of the 51 states have an observation each period)
_coverage = _wide_all.notna().sum(axis=1).rename("n_states_reporting").to_frame()
_coverage.index.name = "period"
print("V1 - values reported per period, REVISED (current) data (no missing fields; counts states present):\n")
print("distribution over all periods (how many periods have N states reporting):")
print(_coverage["n_states_reporting"].value_counts().sort_index().rename("n_periods").to_frame().T.to_string())
_incomplete = _coverage[_coverage["n_states_reporting"] < len(_states_sorted)]
print(f"\nincomplete periods (< {len(_states_sorted)} states): {len(_incomplete)} of {len(_cal)}")
if len(_incomplete):
    print(_incomplete.head(40).to_string())
else:
    print("  (complete panel - every period has all states)")

# (3) duplicate (state, period) keys
_dupmask = _prim.duplicated(["state", "obs_date"], keep=False)
print(f"\nV1 - duplicate (state, period) keys, REVISED (current) data: {int(_dupmask.sum())}")
if int(_dupmask.sum()):
    print(_prim[_dupmask].groupby(["state", "obs_date"]).size().rename("count").reset_index().to_string(index=False))
else:
    _cnt = _prim.groupby("state")["obs_date"].agg(n_obs="size", n_unique_periods="nunique")
    print("no duplicates - every (state, period) is unique. Per-state observation counts:")
    print(_cnt.describe().round(1).to_string())

In [ ]:
# ── V1.4/V1.4b - are zero (<=0) values ever revised to non-zero? ──────────────
# For every observation that is <= 0 in either the first-release or the revised (current) data, pull
# its full ALFRED trajectory (output_type=2) and lay out the value at each successive release date, so
# we can see whether a zero was later corrected upward. Each rel* column is 'release_date=value'.
# Where no vintage is archived (pre-archive obs), the first column falls back to the current value.
_prim = ur_c.copy()
_vint = df_v.copy()
_zero_cur = _prim.loc[_prim["value_current"] <= 0, ["state", "obs_date"]]
_zero_first = (_vint.loc[_vint["value_first_release"] <= 0, ["state", "obs_date"]]
               if "value_first_release" in _vint.columns else _prim.iloc[0:0][["state", "obs_date"]])
_zero_keys = pd.concat([_zero_cur, _zero_first]).drop_duplicates().sort_values(["state", "obs_date"])
_SID = {name: f"{ab}UR" for name, ab, fp in STATES}
_curmap = _prim.set_index(["state", "obs_date"])["value_current"]

_rows = []
for _st in sorted(_zero_keys["state"].unique()):
    try:
        _d = fred_get({"series_id": _SID[_st], "realtime_start": "1776-07-04", "realtime_end": "9999-12-31",
                       "output_type": 2, "limit": 100000})
        _t = pd.DataFrame(_d["observations"]).set_index("date")
        _vc = [c for c in _t.columns if c.startswith(_SID[_st])]
        _vals = _t[_vc].replace(".", np.nan).astype(float)
        _vals.columns = pd.to_datetime([c.split("_", 1)[1] for c in _vc])
    except Exception:
        _vals = None
    for _od in sorted(_zero_keys.loc[_zero_keys["state"] == _st, "obs_date"]):
        _key = pd.Timestamp(_od).strftime("%Y-%m-%d")
        _row = {"state": _st, "obs_date": _key}
        if _vals is not None and _key in _vals.index:
            _s = _vals.loc[_key].dropna()
            _steps = _s[_s != _s.shift()]                       # keep only where the value changed
            for _i, (_rd, _v) in enumerate(_steps.items(), start=1):
                _row[f"rel{_i}"] = f"{_rd.date()}={_v:g}"
            _row["ever_nonzero"] = bool((_s != 0).any())
        else:
            _cv = _curmap.get((_st, _od), np.nan)
            _row["rel1"] = f"current={_cv:g}"                    # vintage not archived -> current value
            _row["ever_nonzero"] = bool(pd.notna(_cv) and _cv > 0)
        _rows.append(_row)

_ztable = pd.DataFrame(_rows)
print("V1.4/V1.4b - zero-value revision trajectory (first release -> successive vintages):")
print("columns rel1, rel2, ... are 'release_date=value'; rel1 is the earliest archived release.\n")
if len(_ztable):
    print(_ztable.fillna("").to_string(index=False))
    print(f"\n{int(_ztable['ever_nonzero'].sum())} of {len(_ztable)} zero observations were non-zero in some "
          f"vintage (i.e. later revised away from zero).")
else:
    print("  (no zero / non-positive values in this dataset - nothing to track)")

## V2 - Calendar & frequency (observation clock)

Validates the **observation clock**: `obs_date` stamps the reference month (FRED uses the 1st; the rate reflects the week containing the 12th).

| Check | Question it answers |
|---|---|
| V2.1 | Are all observation dates month-starts? |
| V2.2 | Are there missing months in any state's history? |
| V2.3 | Do all states cover the same window? |

In [ ]:
non_month_start = int((df_c["obs_date"].dt.day != 1).sum())
add_check("V2", "V2.1", "All obs_date are month-starts (monthly reference convention)",
          "PASS" if non_month_start == 0 else "FAIL",
          metric=f"{non_month_start} non-first-of-month", threshold="== 0")

# Interior monthly gaps. A gap shared by ALL states in the same month is a systematic BLS
# data-availability event (documented WARN), not a per-state data error (FAIL).
miss_by_month = {}
for state, g in ur_c.groupby("state"):
    cal = pd.date_range(g["obs_date"].min(), g["obs_date"].max(), freq="MS")
    for d in cal.difference(g["obs_date"]):
        miss_by_month[d] = miss_by_month.get(d, 0) + 1
systematic = {d: n for d, n in miss_by_month.items() if n >= 45}      # affects (nearly) all states
idiosyncratic = {d: n for d, n in miss_by_month.items() if n < 45}
if not miss_by_month:
    res = "PASS"
elif not idiosyncratic:
    res = "WARN"
else:
    res = "FAIL"
add_check("V2", "V2.2", "Missing months: systematic (shared) vs idiosyncratic (per-state)",
          res,
          metric=(f"{len(systematic)} month(s) missing across all states: "
                  + ", ".join(d.strftime("%Y-%m") for d in sorted(systematic))
                  + (f"; {len(idiosyncratic)} idiosyncratic" if idiosyncratic else "")),
          threshold="0 idiosyncratic gaps; shared BLS-wide holes documented",
          note="Oct-2025 is absent for every state AND for national UNRATE - a BLS data-availability event "
               "(the late-2025 federal shutdown delayed the release). Not a fetch error; the PIT availability "
               "frontier skips it automatically. RULE: do not interpolate the rate across this hole.")

starts = ur_c.groupby("state")["obs_date"].min()
ends   = ur_c.groupby("state")["obs_date"].max()
add_check("V2", "V2.3", "Panel edges aligned across states",
          "PASS" if starts.nunique() == 1 and ends.nunique() == 1 else "WARN",
          metric=f"start {starts.min().date()}..{starts.max().date()}, end {ends.min().date()}..{ends.max().date()}",
          threshold="single common start and end",
          note="balanced monthly panel 1976-01 onward")

## V3 - Point-in-time & release calendar (information clock)

| Clock | Field | Convention | Validated by |
|---|---|---|---|
| **Observation time** - the month *measured* | `obs_date` | 1st of the reference month (week incl. 12th) | V2.1 |
| **Information time** - when it became *known* | `first_release_date` / vintage date | ~50 days after month end, Friday, joint state emp/unemp report | V3.2-V3.4 |

| Check | Question it answers |
|---|---|
| V3.1 | How far back can point-in-time be reconstructed? |
| V3.2 | Does the national vintage calendar match the jobs-report schedule? |
| V3.3 | Is the state publication lag ~50 days? |
| V3.4 | Do state releases follow the national jobs report? |
| V3.5 | Is the publication-lag regime stable over time? |
| V3.6 | Do months become available in order? |
| V3.7 | Does the PIT panel contain any look-ahead? |

In [ ]:
vint_start = df_v["first_release_date"].min()
add_check("V3", "V3.1", "ALFRED vintage boundary identified", "WARN",
          metric=f"first vintage {vint_start.date()}", threshold="documented",
          note="pre-boundary history exists only as current (revised) values: usable for climatology, "
               "NOT for point-in-time backtesting")

wd_names = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}
nat_wd = nat_vdates.dt.dayofweek.value_counts(normalize=True).sort_index()
add_check("V3", "V3.2", "National UNRATE vintages land on the jobs-report day (Friday)",
          "PASS" if nat_wd.get(4, 0) >= 0.80 else "WARN",
          metric=", ".join(f"{wd_names[k]} {v:.0%}" for k, v in nat_wd.items() if v >= 0.01),
          threshold="Friday >= 80%",
          note="the national Employment Situation is released the first Friday after the reference month")

state_lag_med = df_v["pub_lag_days"].median()
add_check("V3", "V3.3", "State publication lag ~50 days (joint state emp/unemp report)",
          "PASS" if 40 <= state_lag_med <= 60 else "WARN",
          metric=f"median {state_lag_med:.0f}d (IQR {df_v['pub_lag_days'].quantile(.25):.0f}-{df_v['pub_lag_days'].quantile(.75):.0f})",
          threshold="median in [40, 60] days",
          note="released together with state nonfarm payrolls, ~3 weeks after the national jobs report")

nat_rel = nat_v.set_index("obs_date")["first_release_date"]
al = df_v[["state", "obs_date", "first_release_date", "pub_lag_days"]].copy()
al["nat_release"] = al["obs_date"].map(nat_rel)
al = al.dropna(subset=["nat_release"])
al["gap_days"] = (al["first_release_date"] - al["nat_release"]).dt.days
after = (al["gap_days"] > 0).mean()
add_check("V3", "V3.4", "State UR published after the national jobs report",
          "PASS" if after >= 0.95 else "WARN",
          metric=f"{after:.1%} of state months released after national; median gap +{al['gap_days'].median():.0f}d",
          threshold=">= 95% after national",
          note="state detail follows the national aggregate; a state rate is never available before the "
               "national UNRATE for the same month")

In [ ]:
df_v["rel_year"] = df_v["first_release_date"].dt.year
lag_by_year = df_v.groupby("rel_year")["pub_lag_days"].median()
lag_range = lag_by_year.max() - lag_by_year.min()
add_check("V3", "V3.5", "Publication-lag regime stable over time",
          "PASS" if lag_range <= 7 else "WARN",
          metric=f"median lag by year spans {lag_by_year.min():.0f}-{lag_by_year.max():.0f}d (range {lag_range:.0f}d)",
          threshold="yearly median range <= 7 days",
          note="a stable lag lets a backtest assume a single availability rule")

inversions = 0
for state, g in df_v.groupby("state"):
    rel = g.sort_values("obs_date")["first_release_date"].values
    inversions += int((rel[1:] < rel[:-1]).sum())
add_check("V3", "V3.6", "Release dates weakly increasing in obs order (per state)",
          "PASS" if inversions == 0 else "WARN",
          metric=f"{inversions} inversions", threshold="== 0",
          note="the PIT builder uses the cummax release frontier, so small inversions are harmless")

rel_lookup = df_v.set_index(["state", "obs_date"])["first_release_date"]
keys = pd.MultiIndex.from_arrays([pit["state"], pit["ur_latest_obs"]])
pit_rel = rel_lookup.reindex(keys).values
violations = int((pd.to_datetime(pit_rel) > pit["as_of_date"]).sum())
unmatched = int(pd.isna(pit_rel).sum())
add_check("V3", "V3.7", "PIT panel look-ahead guard (ur_latest_obs released <= as_of_date)",
          "PASS" if violations == 0 and unmatched == 0 else "FAIL",
          metric=f"{violations} violations, {unmatched} unmatched of {len(pit):,} rows", threshold="== 0")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(lag_by_year.index, lag_by_year.values, marker="o", color="#1f3864")
ax.axhspan(40, 60, color="steelblue", alpha=0.10)
ax.set(title="State UR median publication lag by release year", ylabel="days after month end", xlabel="release year")
plt.tight_layout(); plt.show()

## V4 - Revision behavior (moderate & quantized)

U3 sits between claims (never revised) and NFP (always revised): most revisions are small and, because the rate is published to 0.1pp, many round to exactly zero. The annual benchmark still drives the larger ones.

| Check | Question it answers |
|---|---|
| V4.1 | How often, and by how much, does a first print get revised? |
| V4.2 | Do the larger revisions cluster at the annual benchmark? |
| V4.3 | Is there a systematic bias in first prints? |
| V4.4 | How large is the national UNRATE revision (calibration)? |

In [ ]:
# V4.1 endpoint revision: first release vs current (percentage POINTS, since it's a rate)
rev = df_v.merge(ur_c[["state", "obs_date", "value_current"]], on=["state", "obs_date"], how="inner")
rev["revision"] = rev["value_current"] - rev["value_first_release"]     # percentage points
share_rev = (rev["revision"].abs() > 0.05).mean()
add_check("V4", "V4.1", "Endpoint revision rate & magnitude (first release vs current, pp)",
          "WARN" if share_rev > 0.05 else "PASS",
          metric=f"{share_rev:.0%} of months revised; exact-zero {100*(rev['revision']==0).mean():.0f}%, "
                 f"within +-0.1pp {100*(rev['revision'].abs()<=0.1).mean():.0f}%, within +-0.5pp "
                 f"{100*(rev['revision'].abs()<=0.5).mean():.0f}%, max |{rev['revision'].abs().max():.1f}|pp",
          threshold="informational - U3 is moderately, quantized-revised",
          note="between claims (~0%) and NFP (~99%): most revisions are within the 0.1pp rounding, but "
               "benchmark years and COVID produce >1pp moves - so backtests still need vintage data")

rev["year"] = rev["obs_date"].dt.year
print("\nLargest revisions (first -> current, percentage points):")
cols = ["state", "obs_date", "first_release_date", "value_first_release", "value_current", "revision"]
top = rev.nlargest(8, "revision", keep="all").head(8)[cols].copy()
top["obs_date"] = top["obs_date"].dt.strftime("%Y-%m")
top["first_release_date"] = top["first_release_date"].dt.strftime("%Y-%m-%d")
print(top.to_string(index=False))
rev.to_csv("ur_revisions.csv", index=False)
print("\nFull revision table saved to ur_revisions.csv")

In [ ]:
# V4.2 benchmark clustering via full vintage trajectories (output_type=2) for sample states
BENCH_SAMPLE = {"CAUR": "California", "TXUR": "Texas", "NYUR": "New York"}
bench_months = []
for sid, name in BENCH_SAMPLE.items():
    d = fred_get({"series_id": sid, "realtime_start": "1776-07-04", "realtime_end": "9999-12-31",
                  "output_type": 2, "limit": 100000})
    t = pd.DataFrame(d["observations"]).set_index("date")
    vcols = [c for c in t.columns if c.startswith(sid)]
    vals = t[vcols].replace(".", np.nan).astype(float)
    vals.columns = pd.to_datetime([c.split("_", 1)[1] for c in vcols])
    for obs in vals.index:
        row = vals.loc[obs].dropna()
        if len(row) < 2:
            continue
        steps = row.diff().abs()
        if steps.max() > 0:
            bench_months.append(steps.idxmax().month)
    time.sleep(0.3)
bm = pd.Series(bench_months).value_counts().sort_index()
winter_share = bm.reindex([1, 2, 3, 4]).sum() / bm.sum()
add_check("V4", "V4.2", "Larger revisions cluster at the annual benchmark (winter release months)",
          "PASS" if winter_share >= 0.5 else "WARN",
          metric=f"{winter_share:.0%} of largest revisions land in Jan-Apr; modal release month "
                 f"= {bm.idxmax()} ({bm.max()} of {bm.sum()} obs across CA/TX/NY)",
          threshold=">= 50% of biggest revisions in the winter benchmark window",
          note="the annual LAUS re-estimation/re-seasonal-adjustment lands each winter; apply it on the "
               "release date, not spread backward")

month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(9.5, 3.2))
ax.bar([month_names[m-1] for m in bm.index], bm.values,
       color=["#1f3864" if m in (1,2,3,4) else "steelblue" for m in bm.index])
ax.set(title="Release month of each obs-month's LARGEST revision (CA/TX/NY) - annual benchmark", ylabel="count")
plt.tight_layout(); plt.show()

In [ ]:
# V4.3 directional bias
bias = rev["revision"].mean()
add_check("V4", "V4.3", "Systematic bias in first prints",
          "PASS" if abs(bias) < 0.03 else "WARN",
          metric=f"mean revision {bias:+.3f}pp (first prints {'understate' if bias>0 else 'overstate'} on average)",
          threshold="|mean revision| < 0.03pp to treat as unbiased",
          note="a small persistent sign is expected from benchmark direction; large would mean first prints "
               "are predictably off")

# V4.4 national UNRATE revision calibration
nat_rev = nat_v.merge(nat_cur[["obs_date", "value"]].rename(columns={"value": "current"}), on="obs_date", how="inner")
nat_rev["rev"] = nat_rev["current"] - nat_rev["value"]
add_check("V4", "V4.4", "National UNRATE revision benchmark quantified",
          "PASS",
          metric=f"median {nat_rev['rev'].median():+.2f}pp, IQR [{nat_rev['rev'].quantile(.25):+.2f}, "
                 f"{nat_rev['rev'].quantile(.75):+.2f}]pp, max |{nat_rev['rev'].abs().max():.1f}|pp",
          threshold="informational calibration",
          note="the national rate is revised on the same schedule; its magnitude anchors state expectations")

# ── visual: revision distribution (percentage points) ─────────────────────────
fig, ax = plt.subplots(figsize=(9.5, 3.2))
ax.hist(rev["revision"].clip(-1, 1), bins=41, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", ls="--", lw=1)
ax.set(title="UR first release -> current revision (percentage points, clipped +-1pp)",
       xlabel="revision (pp)", ylabel="count")
ax.text(0.02, 0.9, f"exact zero: {100*(rev['revision']==0).mean():.0f}%", transform=ax.transAxes, fontsize=8)
plt.tight_layout(); plt.show()

## V5 - Outliers, breaks & anomalies

| Check | Question it answers |
|---|---|
| V5.1 | Which extreme monthly moves are real (COVID) vs unexplained? |
| V5.2 | Are there stale/frozen stretches (repeated identical rates)? |
| V5.3 | Is the COVID spike present and correctly sized? |

In [ ]:
wide = ur_c.pivot(index="obs_date", columns="state", values="value_current").sort_index()
dchg = wide.diff()   # month-over-month change in pp (natural for a rate)

# V5.1 extreme monthly moves by ABSOLUTE pp change. (Robust-z misbehaves on a 0.1pp-quantized rate:
# the median monthly move is ~0.1pp, so any recession-sized 0.6pp move scores z>6. An absolute pp
# threshold is the meaningful, interpretable extreme for a bounded rate.)
EXTREME_PP = 1.5
ext = dchg.abs() > EXTREME_PP
out_rows = [{"state": st, "obs_date": dt, "chg": dchg.loc[dt, st]}
            for st in wide.columns for dt in dchg.index[ext[st].fillna(False)]]
outliers = pd.DataFrame(out_rows)
outliers["covid"] = outliers["obs_date"].between("2020-03-01", "2020-08-01")
unexplained = outliers[~outliers["covid"]]
add_check("V5", "V5.1", f"Extreme monthly moves classified (|MoM change| > {EXTREME_PP}pp)",
          "PASS" if len(unexplained) / (wide.shape[0]*wide.shape[1]) < 0.003 else "WARN",
          metric=f"{len(outliers)} extreme moves: {int(outliers['covid'].sum())} COVID, {len(unexplained)} other",
          threshold="non-COVID extremes < 0.3% of panel",
          note="the 2020 spike/rebound dominates; the handful of non-COVID extremes are genuine early-1980s "
               "and 2008-09 recession months - regime-flag rather than winsorize away")
if len(unexplained):
    print(unexplained.reindex(unexplained["chg"].abs().sort_values(ascending=False).index)
          .head(10)[["state", "obs_date", "chg"]].to_string(index=False))

# V5.2 long flat stretches. A 0.1pp-quantized rate repeats often (a stable state sits at one value
# for months), so short flat runs are NORMAL - only a >= 12-month (full-year) flat run is worth a look.
stale = []
for state, g in ur_c.groupby("state"):
    g = g.sort_values("obs_date")
    run = (g["value_current"] != g["value_current"].shift()).cumsum()
    for _, r in g.groupby(run):
        if len(r) >= 12:
            stale.append({"state": state, "start": r["obs_date"].min().date(),
                          "months": len(r), "rate": r["value_current"].iloc[0]})
add_check("V5", "V5.2", "No implausibly long flat stretches (>= 12 identical monthly rates)",
          "PASS" if len(stale) < 5 else "WARN",
          metric=f"{len(stale)} runs of >= 12 identical months",
          threshold="< 5 (short flats are normal for a quantized rate)",
          note="short repeats are expected at 0.1pp granularity; a 12+ month flat run flags a very "
               "low-volatility state or a data hold - inspect the listed cases, do not auto-clean")
if stale:
    print(pd.DataFrame(stale).sort_values("months", ascending=False).head(8).to_string(index=False))

# V5.3 COVID spike present and plausibly sized
covid_peak = wide.loc["2020-04-01":"2020-05-01"].max()
add_check("V5", "V5.3", "COVID unemployment spike present and correctly sized",
          "PASS" if covid_peak.median() > 12 and covid_peak.max() < 35 else "WARN",
          metric=f"Apr-May 2020 peak: median {covid_peak.median():.1f}%, max {covid_peak.max():.1f}% ({covid_peak.idxmax()})",
          threshold="median peak > 12%, max < 35% (Nevada ~28-30%)",
          note="a sanity check that the largest real event is present and not truncated/mis-scaled")

## V6 - Cross-sectional coherence (labor-force-weighted)

**The key adaptation for a rate:** you cannot sum unemployment rates. The correct national reconciliation is the **labor-force-weighted mean** of state rates - and the check demonstrates that the *unweighted* mean is biased.

| Check | Question it answers |
|---|---|
| V6.1 | Does the labor-force-weighted mean of state rates track national UNRATE? |
| V6.2 | Does every state co-move with the national unemployment cycle? |

In [ ]:
# V6.1 labor-force-weighted reconciliation
lf_wide = lf_c.pivot(index="obs_date", columns="state", values="value_current").sort_index()
common_cols = wide.columns.intersection(lf_wide.columns)
idx = wide.index.intersection(lf_wide.index)
u = wide.loc[idx, common_cols]; w = lf_wide.loc[idx, common_cols]
full = u.notna().all(axis=1) & w.notna().all(axis=1)
wmean = (u[full] * w[full]).sum(axis=1) / w[full].sum(axis=1)   # labor-force-weighted
smean = u[full].mean(axis=1)                                    # naive unweighted
recon = pd.DataFrame({"weighted": wmean, "unweighted": smean}).join(
    nat_cur.set_index("obs_date")["value"].rename("national"), how="inner").dropna()
w_err = (recon["weighted"] - recon["national"]).abs().median()
s_err = (recon["unweighted"] - recon["national"]).abs().median()
add_check("V6", "V6.1", "Labor-force-weighted mean of state rates tracks national UNRATE",
          "PASS" if w_err < 0.2 else "WARN",
          metric=f"median |weighted - national| = {w_err:.2f}pp  vs  unweighted {s_err:.2f}pp",
          threshold="weighted error < 0.2pp (and clearly better than unweighted)",
          note="rates cannot be summed; small high-unemployment states bias the naive mean, so weighting by "
               "labor force is required to reproduce the national aggregate")

# V6.2 cross-state co-movement (YoY pp change)
yoy = wide.diff(12).dropna(how="all")
cmat = yoy.corr()
mean_corr = cmat.where(~np.eye(len(cmat), dtype=bool)).mean()
low = mean_corr[mean_corr < 0.30]
add_check("V6", "V6.2", "Cross-state co-movement (mean pairwise corr of YoY pp change)",
          "PASS" if len(low) == 0 else "WARN",
          metric=f"panel median {mean_corr.median():.2f}; " + (f"low: {dict(low.round(2))}" if len(low) else "none < 0.30"),
          threshold="every state's mean corr >= 0.30",
          note="unemployment cycles are highly synchronised; a decorrelated state is a data suspect")

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(recon.index, recon["national"], lw=1.6, color="#1f3864", label="national UNRATE")
ax.plot(recon.index, recon["weighted"], lw=1.0, color="seagreen", label="LF-weighted state mean")
ax.plot(recon.index, recon["unweighted"], lw=1.0, color="darkorange", ls="--", label="unweighted mean (biased)")
ax.set(title="State-rate aggregation vs national UNRATE - weighting is required", ylabel="unemployment rate (%)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## V7 - Seasonality & stationarity (forecasting-critical)

SA data, so seasonality should be **absent**. The rate is **cyclical and bounded** (mean-reverting long-run, no trend) - the contrast with trending NFP: the level is highly persistent but does not require differencing to be bounded, though monthly changes are the stationary transform.

| Check | Question it answers |
|---|---|
| V7.1 | Did seasonal adjustment leave the rate free of residual seasonality? |
| V7.2 | Is the rate persistent but mean-reverting (bounded, not trending)? |
| V7.3 | Are monthly pp changes stationary and mildly persistent? |

In [ ]:
SAMPLE = ["California", "Texas", "New York", "Florida", "Ohio"]

# V7.1 residual seasonality: month-of-year R^2 of the pp change, pre-COVID (should be near zero for SA)
def seasonal_r2(series):
    g = series.dropna().diff().dropna()
    g = g[g.index < "2020-01-01"]
    m = pd.get_dummies(g.index.month).values.astype(float)
    yhat = m @ np.linalg.lstsq(m, g.values, rcond=None)[0]
    ss_res = ((g.values - yhat) ** 2).sum(); ss_tot = ((g.values - g.values.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot
r2_sa = np.mean([seasonal_r2(wide[s]) for s in SAMPLE])
add_check("V7", "V7.1", "Seasonally-adjusted rate free of residual seasonality",
          "PASS" if r2_sa < 0.10 else "WARN",
          metric=f"month-of-year R2 of MoM pp change (SA): {r2_sa:.3f}",
          threshold="R2 < 0.10 (little month structure left)",
          note="strong residual seasonality would mean the published SA is inadequate at the monthly frequency")

# V7.2 persistence & mean-reversion of the level (bounded, not trending)
from statsmodels.tsa.stattools import adfuller
rows = []
for s in SAMPLE:
    lvl = wide[s].dropna()
    p_lvl = adfuller(lvl, autolag="AIC")[1]
    ar1 = lvl.diff().dropna()
    rows.append({"state": s, "ar1_level": lvl.autocorr(1), "p_level_adf": p_lvl})
pers = pd.DataFrame(rows)
add_check("V7", "V7.2", "Rate is highly persistent but bounded (cyclical, not trending)",
          "PASS" if pers["ar1_level"].min() > 0.9 else "WARN",
          metric="; ".join(f"{r.state}: AR1(level) {r.ar1_level:.2f}, ADF p {r.p_level_adf:.2f}" for r in pers.itertuples()),
          threshold="AR1(level) > 0.9 for all sample states",
          note="unlike trending NFP, the rate mean-reverts over the cycle - persistent month-to-month but "
               "bounded; model the level with strong AR terms or the pp change")

# V7.3 monthly change stationary & mildly persistent
chg_ok = all(adfuller(wide[s].dropna().diff().dropna(), autolag="AIC")[1] < 0.05 for s in SAMPLE)
ar1_chg = wide.diff().apply(lambda s: s.dropna()[s.dropna().index < "2020-01-01"].autocorr(1))
add_check("V7", "V7.3", "Monthly pp changes stationary with mild momentum",
          "PASS" if chg_ok else "WARN",
          metric=f"all sample changes ADF-stationary: {chg_ok}; median AR1(change) {ar1_chg.median():.2f}",
          threshold="pp changes stationary (p<0.05) for all sample states",
          note="the pp change is the safe stationary transform; mild positive momentum supports change features")

In [ ]:
# ── visual: the bounded, cyclical, no-trend character (selected states) ────────
fig, ax = plt.subplots(figsize=(11, 3.6))
for s in ["California", "Michigan", "Nevada", "North Dakota"]:
    ax.plot(wide.index, wide[s], lw=1.1, label=s)
ax.set(title="State U3 rates - bounded and cyclical (no trend), unlike payroll levels",
       ylabel="unemployment rate (%)")
ax.legend(fontsize=8, ncol=4); plt.tight_layout(); plt.show()

## V8 - Lead-lag readiness

| Check | Question it answers |
|---|---|
| V8.1 | Are the PIT features indexed by information time? |
| V8.2 | Is orientation correct - does the rate move opposite to payroll growth (Okun)? |
| V8.3 | Alignment convention for pairing with monthly indicators. |

In [ ]:
monthly_asof = (pit["as_of_date"].dt.day == 1).all()
lag_med = pit["ur_lag_months"].median()
add_check("V8", "V8.1", "PIT features indexed by information time (monthly as-of grid)",
          "PASS" if monthly_asof and 1 <= lag_med <= 3 else "WARN",
          metric=f"monthly as-of grid: {monthly_asof}; median data-staleness {lag_med:.0f} months",
          threshold="monthly grid; staleness 1-3 months (the ~50-day lag)",
          note="at as-of month T the latest available reference month is ~T-2")

# V8.2 orientation: the rate should move OPPOSITE to payroll growth (fetch state NFP SA)
ABBR = {name: ab for name, ab, _ in STATES}
nfp_rows = []
for s in SAMPLE:
    for o in fred_obs({"series_id": f"{ABBR[s]}NA", "observation_start": "1990-01-01", "frequency": "m"}):
        nfp_rows.append({"state": s, "date": pd.to_datetime(o["date"]), "nfp": float(o["value"])})
    time.sleep(0.3)
nfp = pd.DataFrame(nfp_rows)
signs = []
for s in SAMPLE:
    du = wide[s].diff(12).dropna()                                       # YoY change in the rate (pp)
    gn = np.log(nfp[nfp["state"] == s].set_index("date")["nfp"]).diff(12).dropna()  # YoY payroll growth
    j = du.index.intersection(gn.index)
    signs.append({"state": s, "corr": du.loc[j].corr(gn.loc[j])})
sdf = pd.DataFrame(signs)
n_neg = int((sdf["corr"] < -0.3).sum())
add_check("V8", "V8.2", "Orientation: unemployment rate moves opposite to payroll growth (Okun sign)",
          "PASS" if n_neg >= 4 else "FAIL",
          metric="; ".join(f"{r.state}: corr {r.corr:+.2f}" for r in sdf.itertuples()),
          threshold=">= 4/5 states with corr < -0.3",
          note="validates sign/alignment wiring: rising unemployment <-> falling payroll growth. NOT the analysis")

add_check("V8", "V8.3", "Monthly alignment convention documented", "WARN",
          metric="UR references the week incl. the 12th; released with state NFP",
          threshold="documented",
          note="UR and state NFP share a reference period AND a release date, so they join directly with no "
               "relative timing offset - simpler than pairing across different reports")

## V9 - Backtest readiness

In [ ]:
cov = pit.groupby("as_of_date")["state"].nunique()
full_start = cov[cov == 51].index.min()
holes = cov[(cov.index > full_start) & (cov < 51)]
add_check("V9", "V9.1", "PIT coverage complete after ramp-in",
          "PASS" if len(holes) == 0 else "FAIL",
          metric=f"51/51 from {full_start.date()}; {len(holes)} holes after", threshold="0 holes",
          note=f"usable backtest window starts {full_start.date()}")

months_per_state = ur_c.groupby("state")["obs_date"].nunique()
add_check("V9", "V9.2", "Universe stability (balanced panel)",
          "PASS" if months_per_state.nunique() == 1 else "WARN",
          metric=f"months per state: {months_per_state.min()}-{months_per_state.max()}", threshold="identical")

# V9.3 current-vs-vintage contamination a naive backtest would suffer
cur_lookup = ur_c.set_index(["state", "obs_date"])["value_current"]
keys = pd.MultiIndex.from_arrays([pit["state"], pit["ur_latest_obs"]])
cur_at_latest = cur_lookup.reindex(keys).values
contamination = pd.Series(cur_at_latest - pit["ur_t0"].values).dropna()   # percentage points
add_check("V9", "V9.3", "Current-vs-vintage contamination if PIT were ignored",
          "WARN" if contamination.abs().median() > 0.05 else "PASS",
          metric=f"median |gap| {contamination.abs().median():.2f}pp, p90 {contamination.abs().quantile(.9):.2f}pp, "
                 f"max {contamination.abs().max():.1f}pp",
          threshold="informational - drives the PIT verdict",
          note="smaller than NFP but non-trivial: a backtest on current data would use benchmark-revised rates "
               "unknown in real time. Use vintage features (ur_pit_features.csv)")

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(cov.index, cov.values, lw=1, color="#1f3864")
ax.set(title="States available per as-of month (PIT panel)", ylabel="# states"); plt.tight_layout(); plt.show()

## Extended visual diagnostics

Full plot set for the checks above - release timing, revision structure, cross-section and dynamics - so every result is visual as well as tabular. Because U3 is a bounded rate, revision magnitudes are in percentage points.

In [ ]:
# ── Release & timing ─────────────────────────────────────────────────────────
_wdn = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}

fig, ax = plt.subplots(figsize=(9.5, 3))
ax.hist(df_v["pub_lag_days"], bins=range(int(df_v["pub_lag_days"].min()), int(df_v["pub_lag_days"].max())+2),
        color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(df_v["pub_lag_days"].median(), color="red", ls="--", lw=1,
           label=f"median {df_v['pub_lag_days'].median():.0f}d")
ax.set(title="State UR publication-lag distribution", xlabel="days from reference month to first release", ylabel="count")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

_natrel = nat_v.set_index("obs_date")["first_release_date"]
_al = df_v[["state","obs_date","first_release_date"]].copy()
_al["gap"] = (_al["first_release_date"] - _al["obs_date"].map(_natrel)).dt.days
_al = _al.dropna()
fig, ax = plt.subplots(figsize=(9.5, 3))
ax.hist(_al["gap"].clip(0, 40), bins=41, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(_al["gap"].median(), color="red", ls="--", lw=1, label=f"median +{_al['gap'].median():.0f}d after national")
ax.set(title="State UR release minus national jobs-report date", xlabel="days after national release", ylabel="count")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

_wd = df_v["first_release_date"].dt.dayofweek.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar([_wdn[k] for k in _wd.index], _wd.values, color="steelblue")
ax.set(title="State UR first-release capture weekday", ylabel="count"); plt.tight_layout(); plt.show()

In [ ]:
# ── Revision structure (percentage points) ───────────────────────────────────
_bs = rev.groupby("state")["revision"].apply(lambda s: s.abs().mean()).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(_bs.index)[::-1], list(_bs.values)[::-1], color="steelblue")
ax.set(title="Mean absolute revision by state (top 15)", xlabel="mean |revision| (percentage points)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10.5, 3.2))
ax.scatter(rev["obs_date"], rev["revision"].clip(-2, 2), s=7, alpha=0.3, color="steelblue")
ax.axhline(0, color="gray", lw=0.8)
ax.set(title="UR revision size over time (clipped +-2pp)", ylabel="current minus first (pp)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(nat_rev["rev"].clip(-0.5, 0.5), bins=41, color="steelblue", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="red", ls="--", lw=1)
ax.set(title="National UNRATE first-release -> current revision", xlabel="revision (pp)", ylabel="count")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cross-section ────────────────────────────────────────────────────────────
# all-states rates (spaghetti) - bounded and cyclical
fig, ax = plt.subplots(figsize=(11, 3.6))
for c in wide.columns:
    ax.plot(wide.index, wide[c], lw=0.5, alpha=0.4, color="steelblue")
ax.plot(wide.index, wide.median(axis=1), lw=2, color="black", label="cross-state median")
ax.set(title="All states: U3 unemployment rate - bounded and cyclical (no trend)",
       ylabel="unemployment rate (%)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# cross-state co-movement bars (YoY pp change)
_cm = wide.diff(12).corr()
_cm = _cm.where(~np.eye(len(_cm), dtype=bool)).mean().sort_values()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(_cm)), _cm.values, color="steelblue")
ax.set_xticks(range(len(_cm))); ax.set_xticklabels(_cm.index, rotation=90, fontsize=6)
ax.axhline(0.30, color="red", ls="--", lw=1, label="0.30 floor")
ax.set(title="Cross-state co-movement: mean pairwise corr of YoY rate change", ylim=(0, 1))
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ── Outliers & dynamics ──────────────────────────────────────────────────────
# extreme-move classification over time (COVID vs other)
fig, ax = plt.subplots(figsize=(11, 3.2))
for lab, m, col in [("COVID", outliers["covid"], "crimson"), ("other", ~outliers["covid"], "purple")]:
    sub = outliers[m]
    ax.scatter(sub["obs_date"], sub["chg"].clip(-8, 12), s=14, alpha=0.5, color=col, label=f"{lab} ({len(sub)})")
ax.axhline(0, color="gray", lw=0.6)
ax.set(title="Extreme monthly rate moves (|MoM change| > 1.5pp), classified", ylabel="MoM change (pp)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# persistence bars: AR(1) of the level by state (highly persistent, bounded)
_ar1 = wide.apply(lambda s: s.dropna().autocorr(1)).sort_values()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(_ar1)), _ar1.values, color="steelblue")
ax.set_xticks(range(len(_ar1))); ax.set_xticklabels(_ar1.index, rotation=90, fontsize=6)
ax.axhline(0.90, color="red", ls="--", lw=1, label="0.90")
ax.set_ylim(0.8, 1.0)
ax.set(title="Rate persistence: AR(1) of the level by state (highly persistent)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# Okun cross-correlation: unemployment change (t) vs payroll growth (t+k)
_ks = list(range(-6, 7))
fig, ax = plt.subplots(figsize=(10, 3.4))
for s in SAMPLE:
    du = wide[s].diff(12)
    gn = np.log(nfp[nfp["state"] == s].set_index("date")["nfp"]).diff(12)
    prof = []
    for k in _ks:
        gk = gn.shift(-k)
        j = du.index.intersection(gk.dropna().index)
        prof.append(du.reindex(j).corr(gk.reindex(j)))
    ax.plot(_ks, prof, marker=".", lw=1.2, label=s)
ax.axvline(0, color="gray", ls=":", lw=0.8); ax.axhline(0, color="gray", lw=0.8)
ax.set(title="Okun cross-correlation: unemployment change (t) vs payroll growth (t+k)",
       xlabel="k (months)", ylabel="corr")
ax.legend(fontsize=8, ncol=5); plt.tight_layout(); plt.show()

# feature contamination by as-of year (current vs vintage latest value, pp)
_cur = ur_c.set_index(["state", "obs_date"])["value_current"]
_keys = pd.MultiIndex.from_arrays([pit["state"], pit["ur_latest_obs"]])
_gap = _cur.reindex(_keys).values - pit["ur_t0"].values
_cont = pd.Series(np.abs(_gap), index=pit["as_of_date"].dt.year).groupby(level=0).median()
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.bar(_cont.index.astype(int), _cont.values, color="steelblue")
ax.set(title="Current-vs-vintage gap by as-of year (median |gap|, pp)", ylabel="percentage points")
plt.tight_layout(); plt.show()

## V10 - Reproducibility & governance

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "validated_at": pd.Timestamp.now().isoformat(),
    "source": "FRED/ALFRED - series {ABBR}UR (SA U3 rate), {ABBR}LF (labor force), UNRATE",
    "vintage_params": {"realtime_start": "1776-07-04", "realtime_end": "9999-12-31", "output_type": 4},
    "files": {f: {"sha256": sha256(f), "rows": int(pd.read_csv(f).shape[0])} for f in DATA_FILES.values()},
    "data_ranges": {
        "current_history": [str(df_c["obs_date"].min().date()), str(df_c["obs_date"].max().date())],
        "vintage_window":  [str(df_v["first_release_date"].min().date()), str(df_v["first_release_date"].max().date())],
        "pit_panel":       [str(pit["as_of_date"].min().date()), str(pit["as_of_date"].max().date())],
    },
}
with open("ur_validation_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
add_check("V10", "V10.1", "Manifest written (hashes, params, ranges)", "PASS",
          metric="ur_validation_manifest.json", threshold="file exists",
          note="ALFRED queries with pinned realtime params are reproducible; re-fetch and diff hashes to audit")

## Validation report

In [ ]:
QUESTIONS = {
    "V1.1":"Do we have all 51 states in every table?", "V1.2":"Is (state, series, month) a unique key?",
    "V1.3":"Are key fields complete (no nulls)?", "V1.4":"Are all rates in a plausible bounded range?",
    "V1.5":"Are rates quantized to 0.1pp as published?",
    "V2.1":"Are all observation dates month-starts?", "V2.2":"Are there missing months in any state?",
    "V2.3":"Do all states cover the same window?",
    "V3.1":"How far back can point-in-time be reconstructed?", "V3.2":"Does the vintage calendar match the jobs-report schedule?",
    "V3.3":"Is the state publication lag ~50 days?", "V3.4":"Do state releases follow the national jobs report?",
    "V3.5":"Is the publication-lag regime stable over time?", "V3.6":"Do months become available in order?",
    "V3.7":"Does the PIT panel contain any look-ahead?",
    "V4.1":"How often/how much does a first print get revised?", "V4.2":"Do larger revisions cluster at the benchmark?",
    "V4.3":"Is there a systematic bias in first prints?", "V4.4":"How large is the national revision (calibration)?",
    "V5.1":"Which extreme moves are real (COVID) vs unexplained?", "V5.2":"Are there stale/frozen stretches?",
    "V5.3":"Is the COVID spike present and correctly sized?",
    "V6.1":"Does the LF-weighted mean track national UNRATE?", "V6.2":"Does every state co-move with the national cycle?",
    "V7.1":"Did SA leave the rate free of residual seasonality?", "V7.2":"Is the rate persistent but bounded (not trending)?",
    "V7.3":"Are monthly pp changes stationary?",
    "V8.1":"Are features indexed by information time?", "V8.2":"Does the rate move opposite to payrolls (Okun)?",
    "V8.3":"How to align with monthly indicators?",
    "V9.1":"From which date is the PIT panel complete?", "V9.2":"Is the universe stable?",
    "V9.3":"How badly would current-instead-of-vintage contaminate a backtest?",
    "V10.1":"Can the dataset be reproduced and audited?",
}

report = pd.DataFrame(checks)[["module", "id", "check", "result", "metric", "threshold", "note"]]
report.insert(3, "question", report["id"].map(QUESTIONS).fillna(""))
report.to_csv("ur_validation_report.csv", index=False)

counts = report["result"].value_counts()
print(f"Checks: {len(report)}  |  PASS {counts.get('PASS',0)}  WARN {counts.get('WARN',0)}  FAIL {counts.get('FAIL',0)}\n")
missing_q = report.loc[report["question"] == "", "id"].tolist()
if missing_q:
    print(f"NOTE: checks without a mapped question: {missing_q}")

def color(v):
    return {"PASS": "background-color:#d4edda", "WARN": "background-color:#fff3cd",
            "FAIL": "background-color:#f8d7da"}.get(v, "")
report.style.map(color, subset=["result"]).hide(axis="index")

## Gating decisions (go/no-go)

In [ ]:
n_fail = int((report["result"] == "FAIL").sum())
print("=" * 78)
print(f"GATE STATUS: {'BLOCKED - resolve FAILs first' if n_fail else 'CLEARED with documented caveats'}")
print("=" * 78)
print(f"""
G1 - SCOPE CLEARED
  * Backtesting / PIT work: {full_start.date()} onward (51/51 PIT coverage; strict ALFRED vintages)
  * Climatology / cycle estimation: full 1976+ current history
  * NOT cleared: treating pre-{vint_start.date()} data as point-in-time

G2 - REQUIRED TRANSFORMS (from V7)
  * It is a BOUNDED RATE: features are CHANGES in percentage points (pp), not growth rates
  * The level is cyclical and mean-reverting (NOT trending like NFP) - model the level with strong AR
    terms, or the pp change; monthly pp changes are the safe stationary transform
  * SA already removes seasonality (verified V7.1); regime-flag 2020 (COVID spike to ~15-30%)

G3 - AGGREGATION & POINT-IN-TIME (from V4/V6/V9)
  * Reconcile to national by LABOR-FORCE-WEIGHTED mean, never a simple average (unweighted is biased
    ~{s_err:.1f}pp vs weighted ~{w_err:.2f}pp)
  * Revisions are moderate & quantized to 0.1pp ({share_rev:.0%} of months move); benchmark years and COVID
    exceed 1pp - backtests should use vintage features. Naive current-data gap: median {contamination.abs().median():.2f}pp,
    p90 {contamination.abs().quantile(.9):.2f}pp
  * Apply benchmark revisions on their release date; UR and state NFP share reference period and release day
""")